<a href="https://colab.research.google.com/github/paultanisha047-pixel/credit-risk-scoring-engine/blob/main/04_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install boto3 pandas numpy scikit-learn xgboost lightgbm \
             dagshub mlflow pyarrow imbalanced-learn -q
print("Dependencies installed ✅")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/

In [5]:
from google.colab import userdata
import os, boto3, io, json
import pandas as pd
import numpy as np
import dagshub
import mlflow

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
BUCKET = userdata.get('s3_BUCKET')
os.environ["DAGSHUB_CLIENT_TOKEN"] = "eab6589bc3166ecb055d97598ae9fb1e50a66d04"

# Connect to DagsHub MLflow — all runs tracked remotely
# dagshub.init sets MLFLOW_TRACKING_URI automatically
dagshub.init(
    repo_owner='paultanisha047-pixel',
    repo_name='credit-risk-scoring-engine',
    mlflow=True
)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print("DagsHub MLflow connected ✅")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=3fdfb84b-5522-43ec-a550-f36050a3707e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=51e021aa6b6425c677228b61a1a14ac8c5c99906d7ce2f10f68a941869da4ec6




JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [8]:
s3 = boto3.client('s3')

# Load feature table
obj = s3.get_object(Bucket=BUCKET, Key='processed/fct_loans.parquet')
df  = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Load feature config saved by 03_features.ipynb
obj_cfg = s3.get_object(Bucket=BUCKET, Key='processed/feature_config.json')
config  = json.loads(obj_cfg['Body'].read())

SELECTED   = config['selected_features']
CATEGORICALS = config['categorical_features']
NUMERICS   = config['numeric_features']

print(f"Loaded {len(df):,} rows")
print(f"Using {len(SELECTED)} features")

Loaded 1,345,310 rows
Using 22 features


In [9]:
df_train = df[df['is_train'] == True].copy()
df_test  = df[df['is_train'] == False].copy()

# Separate features and target
X_train = df_train[SELECTED]
y_train = df_train['default_flag']
X_test  = df_test[SELECTED]
y_test  = df_test['default_flag']

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"Train default rate: {y_train.mean():.1%}")
print(f"Test default rate : {y_test.mean():.1%}")

X_train: (826604, 22)
X_test : (518706, 22)
Train default rate: 18.4%
Test default rate : 22.4%


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Numeric pipeline:
# 1. Impute nulls with median (for any remaining nulls not handled in dbt)
# 2. StandardScaler — centers and scales features
#    Required for Logistic Regression, harmless for tree models
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical pipeline:
# 1. Impute nulls with 'missing' string
# 2. OrdinalEncoder — converts string categories to integers
#    XGBoost and LightGBM handle ordinal encoding well
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])

# ColumnTransformer applies the right pipeline to each feature type
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline,     NUMERICS),
    ('cat', categorical_pipeline, CATEGORICALS)
])

print("Preprocessing pipeline built ✅")


Preprocessing pipeline built ✅


In [10]:
from sklearn.metrics import roc_auc_score
import numpy as np

def gini_coefficient(y_true, y_pred_proba):
    """
    Gini = 2 * AUC - 1
    Primary credit model metric — measures rank-ordering ability.
    Target: > 0.40 for a production credit model.
    """
    auc = roc_auc_score(y_true, y_pred_proba)
    return 2 * auc - 1

def ks_statistic(y_true, y_pred_proba):
    """
    KS = max separation between cumulative default and non-default distributions.
    Target: > 0.30 for a production credit model.
    """
    df_ks = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred_proba})
    df_ks = df_ks.sort_values('y_pred', ascending=False).reset_index(drop=True)

    total_defaults     = y_true.sum()
    total_non_defaults = len(y_true) - total_defaults

    df_ks['cum_defaults']     = (df_ks['y_true'] == 1).cumsum() / total_defaults
    df_ks['cum_non_defaults'] = (df_ks['y_true'] == 0).cumsum() / total_non_defaults
    df_ks['ks']               = abs(df_ks['cum_defaults'] - df_ks['cum_non_defaults'])

    return df_ks['ks'].max()

def log_metrics_to_mlflow(y_true, y_pred_proba, prefix='test'):
    """
    Log all credit model metrics to MLflow in one call.
    prefix = 'train' or 'test' to distinguish train vs test metrics.
    """
    auc   = roc_auc_score(y_true, y_pred_proba)
    gini  = gini_coefficient(y_true, y_pred_proba)
    ks    = ks_statistic(y_true, y_pred_proba)

    mlflow.log_metric(f'{prefix}_auc',  round(auc,  4))
    mlflow.log_metric(f'{prefix}_gini', round(gini, 4))
    mlflow.log_metric(f'{prefix}_ks',   round(ks,   4))

    return {'auc': auc, 'gini': gini, 'ks': ks}

print("Metric functions ready ✅")

Metric functions ready ✅


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline  # Ensuring Pipeline is imported
import mlflow.sklearn

# Logistic Regression is the interpretable baseline
# Required for regulatory purposes — regulators want to understand
# what drives the model's decisions before approving it
# We use class_weight='balanced' to handle class imbalance
# without resampling — cleaner and more reproducible

with mlflow.start_run(run_name="logistic_regression"):

    # Log hyperparameters
    mlflow.log_param("model_type",    "logistic_regression")
    mlflow.log_param("C",             0.1)
    mlflow.log_param("class_weight",  "balanced")
    mlflow.log_param("max_iter",      1000)
    mlflow.log_param("train_rows",    len(X_train))
    mlflow.log_param("n_features",    len(SELECTED))

    # Build full pipeline: preprocessor + model
    lr_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(
            C=0.1,              # regularization strength — lower = more regularized
            class_weight='balanced',  # adjusts for class imbalance
            max_iter=1000,      # allow enough iterations to converge
            random_state=42,
            n_jobs=-1           # use all CPU cores
        ))
    ])

    print("Training Logistic Regression...")
    lr_pipeline.fit(X_train, y_train)

    # Get predicted probabilities for positive class (default)
    lr_proba_train = lr_pipeline.predict_proba(X_train)[:, 1]
    lr_proba_test  = lr_pipeline.predict_proba(X_test)[:, 1]

    # Log metrics to MLflow
    train_metrics = log_metrics_to_mlflow(y_train, lr_proba_train, 'train')
    test_metrics  = log_metrics_to_mlflow(y_test,  lr_proba_test,  'test')

    print(f"Train — AUC: {train_metrics['auc']:.4f} | Gini: {train_metrics['gini']:.4f} | KS: {train_metrics['ks']:.4f}")
    print(f"Test  — AUC: {test_metrics['auc']:.4f}  | Gini: {test_metrics['gini']:.4f}  | KS: {test_metrics['ks']:.4f}")

    # Log model artifact to MLflow (stored in DagsHub)
    # Added name="..." to fix the deprecation warning
    # Added skops_trusted_types=True to bypass the security block on your pipeline types
    mlflow.sklearn.log_model(
        sk_model=lr_pipeline,
        name="logistic_regression_model",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.compose._column_transformer._RemainderColsList"
        ]
    )

    lr_run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {lr_run_id}")
    print("Logistic Regression logged to MLflow ✅")

Training Logistic Regression...
Train — AUC: 0.7152 | Gini: 0.4304 | KS: 0.3145
Test  — AUC: 0.7001  | Gini: 0.4002  | KS: 0.2900
Run ID: 7ab5a3765aff4c848d899b7e64845960
Logistic Regression logged to MLflow ✅


In [19]:
import xgboost as xgb
import mlflow.sklearn  # Used for logging the full pipeline
from sklearn.pipeline import Pipeline  # Ensuring Pipeline is explicitly imported

# XGBoost is the industry standard for credit scoring
# scale_pos_weight handles class imbalance by weighting the minority class
# ratio = count(negatives) / count(positives)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

with mlflow.start_run(run_name="xgboost"):

    mlflow.log_param("model_type",        "xgboost")
    mlflow.log_param("n_estimators",      500)
    mlflow.log_param("max_depth",         6)
    mlflow.log_param("learning_rate",     0.05)
    mlflow.log_param("subsample",         0.8)
    mlflow.log_param("colsample_bytree",  0.8)
    mlflow.log_param("scale_pos_weight",  round(scale_pos_weight, 2))

    xgb_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', xgb.XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            use_label_encoder=False,
            eval_metric='auc',
            random_state=42,
            n_jobs=-1,
            verbosity=0
        ))
    ])

    print("Training XGBoost...")
    xgb_pipeline.fit(X_train, y_train)

    xgb_proba_train = xgb_pipeline.predict_proba(X_train)[:, 1]
    xgb_proba_test  = xgb_pipeline.predict_proba(X_test)[:, 1]

    train_metrics = log_metrics_to_mlflow(y_train, xgb_proba_train, 'train')
    test_metrics  = log_metrics_to_mlflow(y_test,  xgb_proba_test,  'test')

    print(f"Train — AUC: {train_metrics['auc']:.4f} | Gini: {train_metrics['gini']:.4f} | KS: {train_metrics['ks']:.4f}")
    print(f"Test  — AUC: {test_metrics['auc']:.4f}  | Gini: {test_metrics['gini']:.4f}  | KS: {test_metrics['ks']:.4f}")

    # Indentation completely fixed here:
    mlflow.sklearn.log_model(
        sk_model=xgb_pipeline,
        name="xgboost_model",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.compose._column_transformer._RemainderColsList",
            "xgboost.core.Booster",
            "xgboost.sklearn.XGBClassifier"
        ]
    )

    xgb_run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {xgb_run_id}")
    print("XGBoost logged to MLflow ✅")

scale_pos_weight: 4.43
Training XGBoost...
Train — AUC: 0.7475 | Gini: 0.4949 | KS: 0.3606
Test  — AUC: 0.7100  | Gini: 0.4200  | KS: 0.3030
Run ID: 4dbf042f288141b69ff4cff7059a54a9
XGBoost logged to MLflow ✅


In [22]:
import lightgbm as lgb
import mlflow.lightgbm

# LightGBM is faster than XGBoost on large datasets
# Uses leaf-wise tree growth vs XGBoost's level-wise
# is_unbalance=True handles class imbalance internally

with mlflow.start_run(run_name="lightgbm"):

    mlflow.log_param("model_type",     "lightgbm")
    mlflow.log_param("n_estimators",   500)
    mlflow.log_param("max_depth",      6)
    mlflow.log_param("learning_rate",  0.05)
    mlflow.log_param("num_leaves",     63)
    mlflow.log_param("is_unbalance",   True)

    lgb_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', lgb.LGBMClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            num_leaves=63,      # controls model complexity
            is_unbalance=True,  # handles class imbalance
            random_state=42,
            n_jobs=-1,
            verbose=-1          # suppress training output
        ))
    ])

    print("Training LightGBM...")
    lgb_pipeline.fit(X_train, y_train)

    lgb_proba_train = lgb_pipeline.predict_proba(X_train)[:, 1]
    lgb_proba_test  = lgb_pipeline.predict_proba(X_test)[:, 1]

    train_metrics = log_metrics_to_mlflow(y_train, lgb_proba_train, 'train')
    test_metrics  = log_metrics_to_mlflow(y_test,  lgb_proba_test,  'test')

    print(f"Train — AUC: {train_metrics['auc']:.4f} | Gini: {train_metrics['gini']:.4f} | KS: {train_metrics['ks']:.4f}")
    print(f"Test  — AUC: {test_metrics['auc']:.4f}  | Gini: {test_metrics['gini']:.4f}  | KS: {test_metrics['ks']:.4f}")


    mlflow.sklearn.log_model(
        sk_model=lgb_pipeline,        # Replace with your actual LightGBM pipeline variable
        name="lightgbm_model",         # Fixes the artifact_path deprecation warning
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.compose._column_transformer._RemainderColsList",
            "collections.OrderedDict",
            "lightgbm.basic.Booster",
            "lightgbm.sklearn.LGBMClassifier"
        ]
    )

    lgb_run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {lgb_run_id}")
    print("LightGBM logged to MLflow ✅")



Training LightGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train — AUC: 0.7474 | Gini: 0.4948 | KS: 0.3609
Test  — AUC: 0.7087  | Gini: 0.4173  | KS: 0.3014
Run ID: 34ee5d036d004a4893cff49ce394ea83
LightGBM logged to MLflow ✅


In [23]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost', 'LightGBM'],
    'Test AUC':  [
        roc_auc_score(y_test, lr_proba_test),
        roc_auc_score(y_test, xgb_proba_test),
        roc_auc_score(y_test, lgb_proba_test)
    ],
    'Test Gini': [
        gini_coefficient(y_test, lr_proba_test),
        gini_coefficient(y_test, xgb_proba_test),
        gini_coefficient(y_test, lgb_proba_test)
    ],
    'Test KS': [
        ks_statistic(y_test, lr_proba_test),
        ks_statistic(y_test, xgb_proba_test),
        ks_statistic(y_test, lgb_proba_test)
    ]
}).round(4)

print("Model Comparison:")
print(results.to_string(index=False))

best_model_name = results.loc[results['Test Gini'].idxmax(), 'Model']
print(f"\nBest model: {best_model_name} ✅")


Model Comparison:
              Model  Test AUC  Test Gini  Test KS
Logistic Regression    0.7001     0.4002   0.2900
            XGBoost    0.7100     0.4200   0.3030
           LightGBM    0.7087     0.4173   0.3014

Best model: XGBoost ✅


In [24]:
import pickle

# Save the best performing model to S3
# FastAPI will load this model to score new loan applications
# We use pickle for simplicity — in production would use MLflow model serving

best_pipeline = {
    'Logistic Regression': lr_pipeline,
    'XGBoost':             xgb_pipeline,
    'LightGBM':            lgb_pipeline
}[best_model_name]

# Serialize model to bytes
model_bytes = pickle.dumps(best_pipeline)

# Upload to S3
s3.put_object(
    Bucket=BUCKET,
    Key='models/best_model.pkl',
    Body=model_bytes
)

# Save model metadata — FastAPI reads this to know which model is live
model_metadata = {
    'model_name':    best_model_name,
    'selected_features': SELECTED,
    'test_gini':     float(results.loc[results['Model']==best_model_name, 'Test Gini']),
    'test_auc':      float(results.loc[results['Model']==best_model_name, 'Test AUC']),
    'test_ks':       float(results.loc[results['Model']==best_model_name, 'Test KS']),
}

s3.put_object(
    Bucket=BUCKET,
    Key='models/model_metadata.json',
    Body=json.dumps(model_metadata, indent=2)
)

print(f"Best model ({best_model_name}) saved to s3://{BUCKET}/models/best_model.pkl ✅")
print(f"Model metadata saved to s3://{BUCKET}/models/model_metadata.json ✅")


/tmp/ipykernel_1310/962984965.py:27: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  'test_gini':     float(results.loc[results['Model']==best_model_name, 'Test Gini']),
/tmp/ipykernel_1310/962984965.py:28: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  'test_auc':      float(results.loc[results['Model']==best_model_name, 'Test AUC']),
/tmp/ipykernel_1310/962984965.py:29: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  'test_ks':       float(results.loc[results['Model']==best_model_name, 'Test KS']),


Best model (XGBoost) saved to s3://credit-risk-engine-prod-2026-tp/models/best_model.pkl ✅
Model metadata saved to s3://credit-risk-engine-prod-2026-tp/models/model_metadata.json ✅
